# Converting between raster formats

`Dataset.to_file(path)` picks the output driver from the **file extension**, so most raster
conversions are one call. NetCDF has its own notebook
([GeoTIFF ↔ NetCDF](geotiff-netcdf.ipynb)); this page covers the rest.

| Target | Extension / call | Notes |
|--------|------------------|-------|
| GeoTIFF | `.tif` | the default raster format |
| ASCII grid | `.asc` | single band only (pass `band=`) |
| Cloud Optimized GeoTIFF | `to_file(driver='COG')` | tiled + overviews |
| Zarr | `to_zarr(...)` / `from_zarr(...)` | chunked, cloud-native (`[lazy]` extra) |

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-convert-'))  # scratch dir for outputs
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
ds.shape, ds.epsg

2026-06-08 22:38:02 | INFO | pyramids.base.config | Logging is configured.


((1, 13, 14), 32618)

## GeoTIFF ↔ ASCII grid

Esri ASCII grids hold a single band, so pass the `band` index. The extension drives the driver.

In [3]:
asc = WORK / 'acc.asc'
ds.to_file(str(asc), band=0)
# Round-trip back to GeoTIFF.
back = Dataset.read_file(str(asc))
tif_again = WORK / 'from_asc.tif'
back.to_file(str(tif_again))
back.shape, Dataset.read_file(str(tif_again)).shape

((1, 13, 14), (1, 13, 14))

## GeoTIFF → Cloud Optimized GeoTIFF (COG)

Pass `driver='COG'` to write a tiled, overview-bearing COG that streams efficiently over HTTP.

In [4]:
cog = WORK / 'acc_cog.tif'
ds.to_file(str(cog), driver='COG')
cog.exists(), cog.stat().st_size > 0

(True, True)

## GeoTIFF ↔ Zarr

Zarr stores the raster as chunked arrays (one file per chunk) with the geobox in the metadata —
ideal for parallel/cloud workflows. Needs the `[lazy]` extra.

In [5]:
store = WORK / 'acc.zarr'
ds.to_zarr(str(store), chunks=(1, 16, 16), mode='w')
sorted(p.name for p in store.iterdir())[:5]

['data', 'spatial_ref', 'x', 'y', 'zarr.json']

In [6]:
# Reopen the store and write it back to GeoTIFF.
from_zarr = Dataset.from_zarr(str(store))
tif_from_zarr = WORK / 'from_zarr.tif'
from_zarr.to_file(str(tif_from_zarr))
from_zarr.epsg, Dataset.read_file(str(tif_from_zarr)).shape

(32618, (1, 13, 14))

## Notes

- `to_file` infers the driver from the extension (`.tif`, `.asc`, …); pass `driver=` to override
  (e.g. `'COG'`).
- ASCII grids are single-band — convert one band at a time.
- See also: [GeoTIFF ↔ NetCDF](geotiff-netcdf.ipynb) and [Vector formats](vector-formats.ipynb).